> 📅 **Date: 2026-09-10**

# 🖼️ **Multi-Modal RAG**

> **Goal:** Understand how RAG can work with multiple modalities such as text, tables, and images, and how a multimodal LLM can use retrieved information from all of them to generate an answer.

> **Prerequisite:** This chapter assumes familiarity with RAG, Retrievers, Embedding Models, Text Splitters, Semi-Structured RAG, and **`MultiVectorRetriever`**.

> **Dependencies:**

```python
!pip install -U langchain
!pip install -U langchain-openai
!pip install -U langchain-community
!pip install -U langchain-classic
!pip install -U langchain-chroma
!pip install -U unstructured[all-docs]
!pip install -U chromadb
!pip install -U pypdf
!pip install -U pillow
!pip install -U pytesseract
```

### **Install All Dependencies**

```python
!pip install -U langchain langchain-openai langchain-community langchain-classic langchain-chroma unstructured[all-docs] chromadb pypdf pillow pytesseract
```

---

# 🧩 **RAG Types**

RAG can be extended depending on the types of information present in the knowledge base.

```text
RAG
→ Text

Semi-Structured RAG
→ Text + Tables

Multi-Modal RAG
→ Text + Tables + Images
```

### **Conceptual Comparison**

| RAG Type | Main Data |
|---|---|
| **Traditional RAG** | Text |
| **Semi-Structured RAG** | Text + Tables |
| **Multi-Modal RAG** | Text + Tables + Images |

> **Key idea:** Multi-Modal RAG retrieves information from multiple content types and passes the relevant representations to a multimodal LLM.

---

# 🖼️ **Why Multi-Modal RAG?**

Real-world documents are not limited to paragraphs.

**A financial report may contain:**

```text
Text
Tables
Charts
Graphs
Images
Diagrams
```

A text-only RAG system may retrieve the surrounding text but miss important visual information.

**For example:**

```text
Question
   ↓
"What does this revenue chart show?"
   ↓
Text-only Retriever
   ↓
May miss the actual chart
```

Multi-Modal RAG can retrieve the image itself and provide it to a multimodal model.

---

# 🧠 **Multi-Modal RAG Architecture**

```text
                    SOURCE DOCUMENT
                           │
          ┌────────────────┼────────────────┐
          ↓                ↓                ↓
         TEXT             TABLE            IMAGE
          │                │                │
          ↓                ↓                ↓
       Chunking        Structured       Image Extraction
          │             Handling              │
          │                │                  ↓
          │                │              Base64
          │                │                  │
          └────────────────┼──────────────────┘
                           ↓
                     Create Summaries
                           ↓
                     Embedding Model
                           ↓
                       Vector DB
                           ↓
                        Retriever
                           ↓
                    Original Content
                           ↓
                   Multimodal LLM
                           ↓
                         Answer
```

---

# 🧱 **Why Base64 for Images?**

An image file is binary data.

To pass an image directly inside a multimodal message, we can encode its binary contents into a text-safe representation such as Base64.

```text
Image File
    ↓
Binary Data
    ↓
Base64 Encoding
    ↓
Base64 String
    ↓
Multimodal Message
    ↓
Vision-capable LLM
```

> **Base64 is a representation of the image data; it is not the image itself. The model receives the image through the multimodal message format.**

---

# 🔎 **Extract Images from PDF**

We use Unstructured to identify and extract different document elements.

In [12]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="/content/International-Conference-Brochure-DEI-2024.pdf",
    extract_images_in_pdf=True,
    infer_table_structure=True,
)

### **Important Parameters**

```text
extract_images_in_pdf=True
→ Extract images from the PDF

infer_table_structure=True
→ Attempt to preserve table structure
```

**The resulting elements can contain:**

```text
Text
Table
Image
Other document elements
```

---

# 🛠️ **Tesseract**

OCR tools such as Tesseract may be required for document processing workflows that need text recognition from images.

### **Linux**

```bash
sudo apt install tesseract-ocr
```

### **Python**

**If you specifically need the Python wrapper:**

```python
!pip install pytesseract
```

### **Windows**

Tesseract itself is an executable, **so installing it is not simply:**

```python
pip install tesseract
```

Instead, install the Tesseract OCR application and add its installation directory to the system **`PATH`**.

> **Important: `pytesseract`** is a Python wrapper around the Tesseract OCR engine. Installing the wrapper does not by itself install the Tesseract executable.

---

# 👁️ **Use a Multimodal LLM**

For the example, we use a vision-capable OpenAI chat model.

In [2]:
import os
from google.colab import userdata

openai = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o"
)

**A multimodal model can receive:**

```text
Text
+
Image
```

in the same message.

---

# 🖼️ **Encode an Image in Base64**

In [4]:
import base64

with open("/content/figures/figure-9-43.jpg", "rb") as f:
    binary = f.read()

base64_image = base64.b64encode(binary).decode("utf-8")

base64_image

'...'

---

# 💬 **Send Image to the Multimodal LLM**

In [5]:
from langchain_core.messages import HumanMessage

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Create a concise and detailed summary of the given image."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"
            }
        }
    ]
)

**Invoke the model:**


In [6]:
response = model.invoke([message])

response.content

'The image features an entrance gate designed in a traditional style, with an archway and a tiled roof. The structure is adorned with geometric patterns and decorative round windows. A sign with text in an Indian script is positioned above the central entrance, flanked by decorative motifs and topped with a red roof. The gate includes golden decorative elements and is surrounded by greenery.'

**Conceptually:**

```text
Image
  ↓
Base64
  ↓
HumanMessage
  ↓
Multimodal LLM
  ↓
Image Summary
```

---

# 🧰 **Create a Reusable Image Encoding Function**

In [7]:
def get_image_base64(image_path):

    with open(image_path, "rb") as f:
        binary = f.read()

    base64_image = base64.b64encode(binary).decode("utf-8")

    return base64_image

**Test:**

In [8]:
get_image_base64(
    "/content/figures/figure-9-41.jpg"
)

'...'

---

# 📝 **Create an Image Summary Function**

In [9]:
def create_image_summary(image_path):

    base64_image = get_image_base64(image_path)

    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": "Create a concise and detailed summary of the given image."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            }
        ]
    )

    response = model.invoke([message])

    return response.content

**Test:**

In [10]:
create_image_summary(
    "/content/figures/figure-9-41.jpg"
)

'The image depicts the entrance of Ramoji Film City, featuring a large, prominent sign with "RAMOJI FILM CITY" written on it. In the foreground, there is a tourist tram with "17" marked on it, which is carrying visitors. The tram is painted in bright colors, matching the vibrant atmosphere of the location. The background includes lush greenery and well-maintained landscaping, enhancing the overall appeal of the area.'

---

# 📁 **Process All Images**

In [16]:
import os

image_base64_list = []
image_summary_list = []

folder_path = "/content/figures"

for img in os.listdir(folder_path):

    image_path = os.path.join(folder_path, img)

    # Encode image
    image_base64_list.append(
        get_image_base64(image_path)
    )

    # Create image summary
    image_summary_list.append(
        create_image_summary(image_path)
    )

    print(image_path)

/content/figures/figure-1-2.jpg
/content/figures/figure-9-43.jpg
/content/figures/figure-9-44.jpg
/content/figures/figure-9-40.jpg
/content/figures/figure-9-39.jpg
/content/figures/figure-9-41.jpg
/content/figures/figure-9-38.jpg


**Check the results:**

In [17]:
len(image_base64_list)

7

In [18]:
len(image_summary_list)

7

**Inspect one summary:**

In [19]:
image_summary_list[0]

'The image is a silhouette of diverse people holding hands, depicting inclusion and community. It includes adults, children, and a person in a wheelchair, emphasizing diversity and support among individuals of different ages and abilities. The background is a textured, light-colored surface.'

---

# 🆔 **Create Image IDs**

Each original image needs a unique ID.

In [20]:
import uuid

image_ids = [
    str(uuid.uuid4())
    for _ in image_summary_list
]

**Conceptually:**

```text
Image 1 → ID 1
Image 2 → ID 2
Image 3 → ID 3
```

**The same ID links:**

```text
Image Summary
      ↓
Document ID
      ↓
Original Base64 Image
```

---

# 📄 **Create Image Summary Documents**

In [21]:
from langchain_core.documents import Document

image_summary_docs = []

for ind, summary in enumerate(image_summary_list):

    doc = Document(
        page_content=summary,
        metadata={
            "doc_id": image_ids[ind]
        }
    )

    image_summary_docs.append(doc)

**Check:**

In [22]:
len(image_summary_docs)

7

---

# 💾 **Store Image Summaries in Vector DB**

The image summaries are the searchable representations.

In [23]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

/tmp/ipykernel_2765/2344273067.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [24]:
embedding_model = OpenAIEmbeddings()

In [25]:
vectordb = Chroma(
    collection_name="conference-brochure",
    embedding_function=embedding_model,
    persist_directory="db"
)

/tmp/ipykernel_2765/7847002.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


In [26]:
docstore = InMemoryStore()

In [27]:
retriever = MultiVectorRetriever(
    vectorstore=vectordb, ## Summaries
    docstore=docstore,  ## Original tables & text
    id_key="doc_id",
    embedding=embedding_model,

)

In [28]:
retriever.vectorstore.add_documents(
    image_summary_docs
)

['052f6dd1-71a0-4f35-a50e-33ad898c6279',
 'cff482fc-bfbe-43c5-96ca-f6b29c2b3705',
 '87ba316f-3b75-4846-8fec-dddc99a808e3',
 '6f8ca8f3-3750-4e8b-b436-0f3062998c33',
 '89fb7762-36b7-4e3a-a247-24f22756e8dd',
 '5c8fa62c-8726-4bc7-9fcf-bfa40688013f',
 '6f26dba5-07e9-46e2-9af0-8c4d226032bc']

**Conceptually:**

```text
Original Image
      ↓
Image Summary
      ↓
Embedding
      ↓
Vector
      ↓
Vector DB
```

---

# 💾 **Store Original Images in InMemoryStore**

The original image is stored separately from its searchable summary.

**The important mapping is:**

```text
Image ID
   ↓
Original Base64 Image
```

**Therefore:**

In [29]:
retriever.docstore.mset(
    list(zip(image_ids, image_base64_list))
)

> ⚠️ **Important:** **`mset()`** expects **`(key, value)`** pairs. The key must be the same **`doc_id`** stored in the vector-store metadata.

**Conceptually:**

```text
Vector DB

image_id
   ↓
image summary


InMemoryStore

image_id
   ↓
base64 image
```

---

# 🔀 **Complete Multi-Vector Image Retrieval**

```text
IMAGE
  │
  ├──────────────→ Base64 Image
  │                     │
  │                     ↓
  │                InMemoryStore
  │
  └──────────────→ Image Summary
                        │
                        ↓
                 Embedding Model
                        │
                        ↓
                    Vector DB
```

**At query time:**

```text
Query
  ↓
Retriever
  ↓
Image Summary Search
  ↓
Image ID
  ↓
InMemoryStore
  ↓
Original Base64 Image
```

---

# 🔍 **Query the Retriever**

**Once the image summaries and original images are stored:**

In [30]:
results = retriever.invoke(
    "Tell me about ramoji film city."
)

results

['...']

**The retriever may return a mixture of:**

```text
Text
Tables
Base64 Images
```

depending on what was indexed and what matches the query.

---

# 🧩 **Multi-Modal Context**

**The retrieved context can therefore contain:**

```text
Text
Table
Image (Base64)
```

The final LLM must receive these in appropriate multimodal message formats.

---

# 🏗️ **Multi-Modal RAG Chain**

**The retrieval pipeline needs to distinguish between:**

```text
Base64 Image
      ↓
Image message

Normal text
      ↓
Text message
```

This is why we need helper functions before constructing the final prompt.

---

# 🧪 **Import Required Utilities**

In [31]:
import base64
import io
import re

from PIL import Image
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough
)

---

# 🔎 **Detect Base64 Strings**

In [32]:
def looks_like_base64(sb):

    return (
        re.match(
            "^[A-Za-z0-9+/]+[=]{0,2}$",
            sb
        )
        is not None
    )

This is only a heuristic.

A valid Base64 string is not automatically an image.

Therefore we also need to inspect the decoded bytes.

---

# 🖼️ **Check Whether Base64 Represents an Image**

In [33]:
def is_image_data(b64data):

    image_signatures = {
        b"\xff\xd8\xff": "jpg",
        b"\x89\x50\x4e\x47\x0d\x0a\x1a\x0a": "png",
        b"\x47\x49\x46\x38": "gif",
        b"\x52\x49\x46\x46": "webp",
    }

    try:

        header = base64.b64decode(b64data)[:8]

        for signature, image_format in image_signatures.items():

            if header.startswith(signature):
                return True

        return False

    except Exception:

        return False

**Conceptually:**

```text
Base64 String
     ↓
Decode
     ↓
Inspect File Signature
     ↓
Image?
 ┌───┴───┐
YES      NO
 ↓        ↓
Image     Text
```

---

# 📐 **Resize Base64 Images**

Large images can increase prompt size.

We can resize them before passing them to the multimodal model.

In [34]:
def resize_base64_image(
    base64_string,
    size=(128, 128)
):

    img_data = base64.b64decode(
        base64_string
    )

    img = Image.open(
        io.BytesIO(img_data)
    )

    resized_img = img.resize(
        size,
        Image.LANCZOS
    )

    buffered = io.BytesIO()

    resized_img.save(
        buffered,
        format=img.format
    )

    return base64.b64encode(
        buffered.getvalue()
    ).decode("utf-8")

**For the final pipeline, a size such as:**

```python
size=(1300, 600)
```

may preserve more useful visual detail than a tiny thumbnail, depending on the image.

---

# ✂️ **Separate Images and Text**

In [35]:
def split_image_text_types(docs):

    b64_images = []
    texts = []

    for doc in docs:

        if isinstance(doc, Document):
            doc = doc.page_content

        if (
            isinstance(doc, str)
            and looks_like_base64(doc)
            and is_image_data(doc)
        ):

            doc = resize_base64_image(
                doc,
                size=(1300, 600)
            )

            b64_images.append(doc)

        else:

            texts.append(doc)

    return {
        "images": b64_images,
        "texts": texts
    }

**The function converts:**

```text
Retrieved Documents
       ↓
┌──────┴──────┐
↓             ↓
Images       Text / Tables
↓             ↓
Base64       Strings
```

---

# 📨 **Create the Multimodal Prompt**

In [36]:
def img_prompt_func(data_dict):

    formatted_texts = "\n".join(
        data_dict["context"]["texts"]
    )

    messages = []

    # Add retrieved images
    if data_dict["context"]["images"]:

        for image in data_dict["context"]["images"]:

            image_message = {
                "type": "image_url",
                "image_url": {
                    "url": (
                        f"data:image/jpeg;base64,{image}"
                    )
                }
            }

            messages.append(
                image_message
            )

    # Add text and table context
    text_message = {
        "type": "text",
        "text": f"""
You are a financial analyst.

You will be given a mixture of text, tables,
and images such as charts or graphs.

Use the provided information to answer the
user's question.

User Question:
{data_dict["question"]}

Text and / or Tables:
{formatted_texts}
"""
    }

    messages.append(text_message)

    return [
        HumanMessage(
            content=messages
        )
    ]

---

# 🔗 **Create the Multi-Modal RAG Chain**

In [37]:
def multi_modal_rag_chain(retriever):

    chain = (
        {
            "context": (
                retriever
                | RunnableLambda(
                    split_image_text_types
                )
            ),

            "question": RunnablePassthrough()
        }

        | RunnableLambda(
            img_prompt_func
        )

        | model

        | StrOutputParser()
    )

    return chain

---

# 🧠 **What Happens Inside the Chain?**

**Suppose the user asks:**

```text
"What does this revenue chart show?"
```

**Execution is:**

```text
User Query
    ↓
Retriever
    ↓
Relevant Chunks
    ↓
split_image_text_types()
    ↓
{
    "images": [...],
    "texts": [...]
}
    ↓
img_prompt_func()
    ↓
HumanMessage(
    text + images
)
    ↓
Multimodal LLM
    ↓
StrOutputParser
    ↓
Answer
```

---

# 🔄 **RunnableLambda**

> **RunnableLambda = Converts a normal Python function into a LangChain Runnable.**

**For example:**

```python
RunnableLambda(
    split_image_text_types
)
```

allows the function to participate in the LCEL pipeline.

**Conceptually:**

```text
Python Function
      ↓
RunnableLambda
      ↓
LangChain Runnable
```

---

# 🔄 **RunnablePassthrough**

> **RunnablePassthrough = Passes the original input through unchanged.**

```python
RunnablePassthrough().invoke(
    "What is Tesla sales?"
)
```

**returns:**

```text
"What is Tesla sales?"
```

In the RAG chain, this preserves the original user question while the retriever independently creates the context.

---

# 🧪 **Run the Multi-Modal RAG Chain**

In [38]:
chain = multi_modal_rag_chain(
    retriever
)

**Then:**

In [41]:
chain.invoke(
    "Give me Registration Fee Details"
)

'The images provided do not contain details about registration fees. Please provide the text information or tables related to the registration fees, and I will be able to assist you further.'

---

# 🧠 **Chain Data Flow**

**The dictionary at the beginning of the chain creates two values from the same input:**

```python
{
    "context": retriever | RunnableLambda(...),
    "question": RunnablePassthrough()
}
```

**Conceptually:**

```text
Input Question
      │
      ├──────────────→ Retriever
      │                     ↓
      │               Relevant Chunks
      │                     ↓
      │          split_image_text_types()
      │                     ↓
      │          images + texts
      │
      └──────────────→ RunnablePassthrough()
                            ↓
                         question
```

**These are then passed to:**

```text
img_prompt_func()
```

**which creates the final:**

```text
HumanMessage
```

**containing:**

```text
Images
+
Text / Tables
+
Question
```

---

# 📊 **Text vs Semi-Structured vs Multi-Modal RAG**

| Architecture | Data | Main Challenge | Typical Strategy |
|---|---|---|---|
| **RAG** | Text | Retrieve useful text | Text embeddings |
| **Semi-Structured RAG** | Text + Tables | Preserve table structure | Structured extraction + summaries |
| **Multi-Modal RAG** | Text + Tables + Images | Retrieve and pass multiple modalities | Multi-vector retrieval + multimodal LLM |

---

# 🧠 **Why MultiVectorRetriever Fits Multi-Modal RAG**

The same principle used for semi-structured RAG works for images.

```text
Image
   ↓
Image Summary
   ↓
Embedding
   ↓
Vector DB
```

**while:**

```text
Image
   ↓
Base64
   ↓
Document Store
```

**This gives us:**

```text
Summary
→ Searchable representation

Original Image
→ Detailed representation for the LLM
```

---

# ✅ **Advantages of Multi-Modal RAG**

```text
Handles text + tables + images
Useful for charts and graphs
Preserves original visual information
Allows summary-based image retrieval
Can support financial and business documents
Extends standard RAG to richer document types
```

---

# ⚠️ **Limitations**

```text
More complex indexing pipeline
Image processing can be expensive
Multimodal model calls may cost more
Images can significantly increase prompt size
OCR / extraction quality can affect results
Requires a multimodal model for visual understanding
```

---

# 💰 **Context and Image Size**

**Large images can increase:**

```text
Prompt Size
   ↓
Token / Image Processing Cost
   ↓
Latency
```

Therefore, resizing images or selecting only the most relevant images can be useful.

```text
Retrieve
  ↓
Select Relevant Images
  ↓
Resize / Compress
  ↓
Send to Multimodal LLM
```

> **Do not aggressively resize images when small text or chart labels are important. Image preprocessing should preserve the visual information required to answer the query.**

---

# 🔐 **Security Note**

Base64 is **encoding, not encryption**.

```text
Base64
→ Data representation

Encryption
→ Data protection
```

Do not assume that converting an image to Base64 makes sensitive information secure.

---

# 🧠 **Key Takeaways**

```text
Multi-Modal RAG
→ Text + Tables + Images

Image
→ Base64 → Summary → Embedding → Vector DB

Original Image
→ Base64 → Document Store

Retriever
→ Finds relevant summaries

Document Store
→ Returns original content

Multimodal LLM
→ Understands text + visual information
```

---

# 🧠 **Ultimate Memory Trick**

```text
TEXT
→ Embed → Retrieve

TABLE
→ Structure → Summarize → Retrieve → Return Original

IMAGE
→ Base64 → Summarize → Retrieve → Return Original Image

MULTI-MODAL RAG
→ Retrieve different content types
→ Build multimodal prompt
→ Give everything relevant to the vision-capable LLM
```

---

# 🎯 **Interview-Friendly Explanation**

> **Multi-Modal RAG extends traditional RAG beyond text. Instead of retrieving only text chunks, it can retrieve text, structured tables, and images such as charts or diagrams. For images, we can create summaries for vector search and keep the original image separately in a document store. During retrieval, we use the shared document ID to recover the original image and then pass the relevant text, tables, and images together to a multimodal LLM to generate the final answer.**

---

# 🏁 **Final Mental Model**

```text
                         MULTI-MODAL RAG
                                │
             ┌──────────────────┼──────────────────┐
             ↓                  ↓                  ↓
           TEXT               TABLE              IMAGE
             │                  │                  │
             ↓                  ↓                  ↓
          Chunking         Structure / HTML     Base64
             │                  │                  │
             └──────────────────┼──────────────────┘
                                ↓
                           SUMMARIZATION
                                ↓
                         EMBEDDING MODEL
                                ↓
                            VECTOR DB
                                ↓
                            RETRIEVER
                                ↓
                          DOCUMENT IDS
                                ↓
                         ORIGINAL STORE
                                ↓
              ┌─────────────────┼─────────────────┐
              ↓                 ↓                 ↓
            TEXT              TABLE             IMAGE
                                                  ↓
                                               Base64
              └─────────────────┬─────────────────┘
                                ↓
                       MULTIMODAL PROMPT
                                ↓
                       VISION-CAPABLE LLM
                                ↓
                              ANSWER
```

> **Remember:**  
> **Use summaries or other retrieval-friendly representations to find the right content, then give the multimodal LLM the original text, tables, and images needed to answer the question.**
